In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Configura a renderização gráfica inline no Jupyter notebook
%matplotlib inline

# Quais gravações pertencem a uma versão mini do EEG2025?

Inspecione o catálogo do desafio R5 sem ler os dados brutos de sinal. A lista
mini define os participantes elegíveis; a contagem de gravações do catálogo depende
da disponibilidade das tarefas. Os dados do desafio são derivados a 100 Hz com filtro
de 0.5–50 Hz e não devem ser confundidos com as gravações originais do HBN no OpenNeuro.


## Antes de começar

Use um ambiente com o EEGDash instalado e uma conexão de internet para acessar
o catálogo. Esta página solicita apenas metadados; não requer GPU nem o download
de sinais. ``EEGDASH_CACHE_DIR`` seleciona um cache reutilizável, cujo padrão é
``~/.eegdash_cache``. Ler o atributo ``.raw`` de uma gravação posteriormente é uma etapa
separada que adquire o sinal e seus arquivos auxiliares (*sidecars*).

A questão é verificar se uma coorte proposta contém os participantes e tarefas
esperados. Não há alvo de predição nem pontuação de treino/teste nesta página.
O ``p_factor`` é solicitado para inspecionar os metadados de participantes disponíveis,
não para construir um novo alvo ou inferir que todo participante possui um valor válido.



In [ ]:
# Importa módulos de sistema operacional e manipulação de caminhos no sistema de arquivos
import os
from pathlib import Path

# Importa a classe do dataset do desafio e o mapeamento dos participantes da versão mini do EEGDash
from eegdash import EEGChallengeDataset
from eegdash.const import SUBJECT_MINI_RELEASE_MAP

## Selecionar o lançamento do desafio

``release="R5"`` seleciona o mapeamento de versões do desafio; ``mini=True``
restringe a elegibilidade à lista curada de participantes. Nenhum filtro de tarefa é usado,
de modo que diversas tarefas e execuções (*runs*) podem pertencer à mesma pessoa. O carregador
do desafio também seleciona o local de armazenamento pré-processado do desafio. Substituí-lo por
uma consulta ao OpenNeuro altera o produto de dados, mesmo quando os IDs dos participantes coincidem.

``description_fields`` solicita campos úteis para verificar a elegibilidade da coorte.
A descrição pode conter metadados adicionais da fonte. Valores ausentes de idade, sexo ou
fator p (*p-factor*) exigem uma política de exclusão explícita em uma análise supervisionada
posterior; esta etapa de descoberta deliberadamente não descarta registros silenciosamente.



In [ ]:
# Inicializa o catálogo do dataset do desafio R5 restrito à lista curada (mini=True)
dataset = EEGChallengeDataset(
    release="R5",
    mini=True,
    cache_dir=Path(
        os.environ.get("EEGDASH_CACHE_DIR", "~/.eegdash_cache")
    ).expanduser(),
    # Solicita os campos de metadados essenciais para inspeção da coorte
    description_fields=["subject", "task", "run", "age", "sex", "p_factor"],
)

## Contar participantes separadamente das gravações

Uma linha de ``description`` descreve uma gravação, não uma pessoa independente.
A lista de elegibilidade, os sujeitos únicos correspondentes e o número de objetos de gravação
respondem, portanto, a perguntas diferentes. Agrupar por tarefa expõe execuções repetidas:
``recordings`` pode exceder ``participants`` sem indicar dados duplicados.

A asserção de subconjunto verifica se os resultados do catálogo respeitam a elegibilidade da versão mini.
Ela não exige que todos os sujeitos elegíveis tenham todas as tarefas, nem fixa uma contagem de
catálogo em tempo real dentro de um teste. Inspecione a tabela impressa antes de escolher uma tarefa
para o próximo tutorial.



In [ ]:
# Extrai o DataFrame com a descrição dos metadados das gravações do dataset
metadata = dataset.description
# Assegura que todos os participantes encontrados pertencem à lista oficial da versão mini R5
assert set(metadata.subject).issubset(SUBJECT_MINI_RELEASE_MAP["R5"])
# Exibe o total de participantes elegíveis no mapeamento oficial
print("Eligible mini participants:", len(SUBJECT_MINI_RELEASE_MAP["R5"]))
# Exibe a quantidade de participantes distintos encontrados no catálogo
print("Matched participants:", metadata.subject.nunique())
# Exibe o total de gravações registradas no catálogo
print("Matched recordings:", len(dataset.datasets))
# Agrupa e exibe o número de gravações e participantes únicos por tarefa
print(
    metadata.groupby("task").agg(
        recordings=("subject", "size"), participants=("subject", "nunique")
    )
)
# Imprime as primeiras linhas da tabela de metadados sem o índice numérico
print(metadata.head().to_string(index=False))

Selecione participantes explicitamente antes de acessar ``.raw``: o construtor acima
descobre metadados, enquanto ``.raw`` dispara a aquisição dos sinais propriamente ditos.



In [ ]:
# Imprime o caminho relativo BIDS do primeiro registro para verificar a proveniência do arquivo
print("First recording provenance:", dataset.records[0]["bids_relpath"])

## Usar o resultado para definir uma coorte delimitada

O ``bids_relpath`` impresso identifica uma gravação de origem concreta. Mantenha
esse caminho, versão, sujeito e tarefa com quaisquer metadados de janelas posteriores. A contagem
de amostras de um modelo deve vir de suas janelas; a contagem de participantes de uma coorte deve
vir de identificadores de sujeitos únicos.

Em seguida, escolha três sujeitos listados e uma única tarefa, adicione esses filtros ao
construtor e inspecione suas anotações por meio de ``.raw``. Estime o download
resultante antes de ampliar a consulta. Para regressão, verifique a disponibilidade do
fator p observado antes da extração de características; para tempo de reação, retenha
os eventos de estímulo e resposta observados. Esta tabela de metadados por si só não
pode estabelecer o desempenho do modelo nem a qualidade dos dados.

Exemplo relacionado de carregamento de dados: [Braindecode BIDS Dataset Example](https://braindecode.org/dev/auto_examples/datasets_io/bids_dataset_example.html).

